# Feature Importance Analysis — Equity (es1s, nq1s, fesx1s)

**Goal**: Cluster correlated features into semantically coherent groups, then measure *cluster-level* importance rather than individual feature importance. Individual permutation importance is unreliable under correlation: when two features share information, permuting one leaves the other intact, so the model continues performing well — the feature appears unimportant even if the *information it carries* is critical. Jointly permuting a correlated cluster breaks the shared information simultaneously, giving an honest importance estimate.

**Data constraints**:
- HMM features come **only** from `harry/new_work` pre-computed CSVs (`features_hmm_vol.csv`, `features_hmm_macro.csv`). No HMM outputs from the main branch.
- CPCV splitter from `harry/new_work/cpcv_search.py` with n_groups=6, k=2 → 15 combinatorial paths.
- Triple-barrier labels from `data/meta/triple_barrier_labels_fixed.csv`.
- All timestamps are forward-filled release-dates (EIA/PMI), not interpolated — holiday rows are kept as-is.

In [ ]:
%matplotlib inline

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import display

warnings.filterwarnings('ignore')

_HERE = Path('').resolve()
_REPO = _HERE.parent.parent.parent
for p in [str(_REPO / 'src'), str(_REPO)]:
    if p not in sys.path:
        sys.path.insert(0, p)

from stml.new_work.feature_importance import (
    load_all_data, build_feature_matrix, apply_hygiene, assign_groups,
    compute_spearman_distance, select_k, get_cluster_labels,
    cluster_representatives, build_cluster_map, run_cpcv_importance,
    dimensionality_reduction_check, _macro_sanity_check, _feature_cols,
    OUTPUTS, RANDOM_SEED, CORR_CLUSTER_PREFIXES,
)
from stml.na_checks import native_returns, wide_returns

ASSET_CLASS  = 'equity'
INSTRUMENTS  = ['es1s', 'nq1s', 'fesx1s']
OUTPUTS.mkdir(parents=True, exist_ok=True)
print(f'Asset class : {ASSET_CLASS}')
print(f'Instruments : {INSTRUMENTS}')
print(f'Outputs     → {OUTPUTS}')

In [ ]:
print('Loading data...')
data = load_all_data()
rets_long = native_returns(data['ohlcv'], kind='log')
wide_rets  = wide_returns(rets_long).sort_index()
print('Data loaded.')
print(f"  Labels   : {len(data['labels']):,} rows  ({data['labels']['instrument'].nunique()} instruments)")
print(f"  HMM-vol  : {len(data['hmm_vol']):,} rows")
print(f"  HMM-macro: {len(data['hmm_macro']):,} rows")

---
## Section 1 — Hygiene Log

1. **Sparse column drop** (>30 % NaN): features structurally absent from the event window are removed as columns rather than silently discarding thousands of rows. `f15_dist_lead_lag` was 96.5 % NaN in the metamodel window and is dropped here.
2. **Warmup trim**: rows with any remaining NaN are dropped (rolling indicators not yet warm).
3. **Near-zero variance** (var < 1e-6): constant columns carry no signal.
4. **Near-perfect twin dedup** (|Spearman ρ| ≥ 0.99): redundant twins distort individual permutation importance.
5. **EIA/PMI cadence check**: if a z-scored weekly/monthly series changes every day it has been interpolated — look-ahead bias.
6. **HMM source confirmation**: only `hmm_vol_*` / `hmm_macro_*` from `harry/new_work` CSVs are used.

In [ ]:
all_results = {}
for inst in INSTRUMENTS:
    print(f'\n{"="*55}\n{inst}\n{"="*55}')
    daily_df, events_df = build_feature_matrix(inst, data, wide_rets)
    hygiene_log = []
    _macro_sanity_check(daily_df, hygiene_log)
    events_df, hygiene_log = apply_hygiene(events_df, daily_df, hygiene_log)
    all_results[inst] = {'daily_df': daily_df, 'events_df': events_df, 'hygiene_log': hygiene_log}
    print(f'  Events  : {len(events_df)}')
    print(f'  Features: {len(_feature_cols(events_df))}')

In [ ]:
for inst in INSTRUMENTS:
    print(f'\n--- Hygiene log: {inst} ---')
    for line in all_results[inst]['hygiene_log']:
        print(' ', line)

---
## Section 2 — Feature Clustering

**Distance**: `sqrt(1 – |ρ|)` Spearman — absolute value so strong negative correlation is treated as closeness. Square root satisfies the triangle inequality.

**Linkage**: Ward (minimises within-cluster variance at each merge).

**K selection**: silhouette on the precomputed distance matrix. Calinski-Harabasz and Davies-Bouldin cross-check.

**Hand-assigned** (not entered into clustering):
| Group | Reason |
|---|---|
| F4_latent | PCA components — orthogonal by construction |
| F5_signal | Discrete {−1, 0, +1} ternary signal features |
| F8_calendar | Deterministic sin/cos — no stochastic variation |

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import squareform

for inst in INSTRUMENTS:
    res = all_results[inst]
    events_df  = res['events_df']
    groups     = assign_groups(events_df)
    corr_cols  = groups['corr_cluster']
    hand_groups = {k: v for k, v in groups.items() if k != 'corr_cluster'}
    X_corr     = events_df[corr_cols].fillna(0)
    dist_mat   = compute_spearman_distance(X_corr)
    best_k, cluster_metrics = select_k(X_corr, dist_mat)
    cluster_labels = get_cluster_labels(dist_mat, best_k)
    cluster_reps   = cluster_representatives(X_corr, cluster_labels)
    cluster_map    = build_cluster_map(corr_cols, cluster_labels, hand_groups)
    res.update({
        'corr_cols': corr_cols, 'hand_groups': hand_groups,
        'dist_mat': dist_mat, 'best_k': best_k,
        'cluster_metrics': cluster_metrics, 'cluster_labels': cluster_labels,
        'cluster_reps': cluster_reps, 'cluster_map': cluster_map,
    })
    print(f'{inst}: {len(corr_cols)} corr-clusterable, K={best_k}, '
          f'{sum(len(v) for v in hand_groups.values())} hand-assigned')
print('\nClustering done.')

In [ ]:
# K-selection metrics — all instruments
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, ['silhouette', 'calinski_harabasz', 'davies_bouldin']):
    for inst in INSTRUMENTS:
        m  = all_results[inst]['cluster_metrics'].set_index('K')
        bk = all_results[inst]['best_k']
        ax.plot(m.index, m[metric], marker='o', label=inst)
        ax.axvline(x=bk, linestyle='--', alpha=0.3)
    ax.set_xlabel('K'); ax.set_title(metric); ax.legend(fontsize=8)
fig.suptitle('K-selection — Spearman Ward clustering', y=1.01)
plt.tight_layout()
fig.savefig(OUTPUTS / 'k_selection_metrics.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Dendrograms — one per instrument
for inst in INSTRUMENTS:
    res      = all_results[inst]
    dist_mat = res['dist_mat']
    best_k   = res['best_k']
    corr_cols = res['corr_cols']
    Z        = linkage(squareform(dist_mat), method='ward')
    cut_h    = Z[-(best_k - 1), 2]

    fig, ax = plt.subplots(figsize=(16, 5))
    dendrogram(Z, labels=corr_cols, ax=ax, leaf_rotation=90,
               leaf_font_size=6, color_threshold=cut_h)
    ax.axhline(y=cut_h, color='red', linestyle='--', linewidth=1.5,
               label=f'K={best_k} cut')
    ax.set_title(f'{inst} — Ward dendrogram  (Spearman distance = sqrt(1–|ρ|))', fontsize=11)
    ax.legend(fontsize=9)
    plt.tight_layout()
    out_dir = OUTPUTS / inst; out_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_dir / 'dendrogram.png', dpi=120)
    plt.show()

In [ ]:
# Explicit cluster contents — every feature listed, per instrument
for inst in INSTRUMENTS:
    cluster_map = all_results[inst]['cluster_map']
    print(f'\n{"="*70}')
    print(f'{inst}  —  {len(cluster_map)} clusters')
    print(f'{"="*70}')

    rows = []
    for cname, cols in cluster_map.items():
        pfx_counts = {}
        for c in cols:
            for p in list(CORR_CLUSTER_PREFIXES) + ['f4_', 'f5_', 'f8_']:
                if c.startswith(p):
                    pfx_counts[p] = pfx_counts.get(p, 0) + 1
                    break
        dom      = max(pfx_counts, key=pfx_counts.get) if pfx_counts else 'other'
        dom_frac = pfx_counts.get(dom, 0) / max(len(cols), 1)
        pure     = '✓ pure' if len(pfx_counts) <= 1 else '✗ mixed'
        feat_str = ', '.join(cols[:8]) + (f'  … +{len(cols)-8} more' if len(cols) > 8 else '')
        rows.append({'cluster': cname, 'n': len(cols),
                     'dominant F-group': dom.rstrip('_'),
                     'dom%': f'{dom_frac:.0%}', 'pure?': pure,
                     'features (first 8)': feat_str})

    display(pd.DataFrame(rows).set_index('cluster'))

    out_dir = OUTPUTS / inst; out_dir.mkdir(parents=True, exist_ok=True)
    member_rows = []
    for cname, cols in cluster_map.items():
        for c in cols:
            member_rows.append({'cluster': cname, 'feature': c})
    pd.DataFrame(member_rows).to_csv(out_dir / 'cluster_membership.csv', index=False)

---
## Section 3 — Per-Feature Importance: The Correlation Problem

When two features A and B are nearly identical, permuting A destroys its values — but the model substitutes B. AUC barely drops, making A look unimportant. Permuting B gives the symmetric result. Neither looks important individually, yet together they carry real predictive information.

**What to look for**: Within a cluster of correlated siblings, individual PFI scores should be small and scattered. The clustered MDA in Section 4 assigns the correct, larger importance to the group as a whole.

**MDI note**: computed on the training set only. Overstates high-cardinality continuous features and features placed early in the tree. Reported as a cross-check, not a primary result.

In [ ]:
# CPCV importance loop — computationally expensive
for inst in INSTRUMENTS:
    res       = all_results[inst]
    events_df = res['events_df']
    feat_cols = _feature_cols(events_df)
    cluster_map = res['cluster_map']
    print(f'\n{inst}: {len(events_df)} events, {len(feat_cols)} features, '
          f'{len(cluster_map)} clusters — running CPCV...')
    importance = run_cpcv_importance(events_df, feat_cols, cluster_map)
    res['importance'] = importance
    res['feat_cols']  = feat_cols
    mean_auc  = np.mean(importance['fold_aucs']) if importance['fold_aucs'] else float('nan')
    shap_mag  = importance['shap_magnitude_mean']
    shap_ok   = len(shap_mag) > 0 and not bool((shap_mag == 0).all())
    shap_info = f'max={shap_mag.max():.4f}  non-zero={int((shap_mag!=0).sum())}/{len(shap_mag)}' if shap_ok else 'ALL-ZERO (check errors above)'
    print(f'  {importance["n_folds"]} CPCV folds, mean AUC = {mean_auc:.3f}  | SHAP: {shap_info}')
print('\nDone.')

In [ ]:
# Per-feature importance bar charts (top 30 by PFI) — es1s
inst_show = 'es1s'
imp = all_results[inst_show]['importance']
pfi_df = pd.DataFrame({'pfi_mean': imp['pfi_mean'], 'pfi_std': imp['pfi_std'],
                        'mdi_mean': imp['mdi_mean'],
                        'shap_magnitude': imp['shap_magnitude_mean'],
                        'shap_signed':    imp['shap_signed_mean']})
pfi_df.index.name = 'feature'
out_dir = OUTPUTS / inst_show; out_dir.mkdir(parents=True, exist_ok=True)
pfi_df.to_csv(out_dir / 'per_feature_importance.csv')

pfi_sorted = pfi_df.sort_values('pfi_mean', ascending=False)
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, col, title in zip(axes,
        ['pfi_mean', 'mdi_mean', 'shap_magnitude'],
        ['PFI — individual AUC drop', 'MDI (train-set bias ⚠)', '|SHAP| mean']):
    s = pfi_sorted[col].head(30)
    ax.barh(range(len(s)), s.values,
            color=['tomato' if v < 0 else 'steelblue' for v in s], alpha=0.8)
    ax.set_yticks(range(len(s))); ax.set_yticklabels(s.index, fontsize=6)
    ax.set_xlabel(title); ax.axvline(x=0, color='black', linewidth=0.8); ax.invert_yaxis()
fig.suptitle(f'{inst_show} — Per-feature importance (top 30 by PFI)', y=1.01)
plt.tight_layout()
fig.savefig(out_dir / 'per_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation problem demo — PFI within the largest cluster for each instrument
hand_labels = {'F4_latent', 'F5_signal', 'F8_calendar'}
for inst in INSTRUMENTS:
    res         = all_results[inst]
    cluster_map = res['cluster_map']
    pfi_mean    = res['importance']['pfi_mean']
    cmda_mean   = res['importance']['clustered_mda_mean']

    corr_clusters = {k: v for k, v in cluster_map.items() if k not in hand_labels}
    biggest = max(corr_clusters, key=lambda k: len(corr_clusters[k]))
    members = corr_clusters[biggest]
    member_pfi = pfi_mean.reindex(members).dropna().sort_values(ascending=False)
    joint_mda  = cmda_mean.get(biggest, float('nan'))

    fig, ax = plt.subplots(figsize=(8, max(3, len(member_pfi) * 0.32)))
    ax.barh(range(len(member_pfi)), member_pfi.values,
            color=['tomato' if v < 0 else 'steelblue' for v in member_pfi.values], alpha=0.8)
    ax.set_yticks(range(len(member_pfi)))
    ax.set_yticklabels(member_pfi.index, fontsize=8)
    ax.set_xlabel('Individual PFI (AUC drop)')
    ax.set_title(f'{inst}  —  cluster "{biggest}"  ({len(members)} siblings)\n'
                 f'Individual PFI near-zero; joint clustered MDA = {joint_mda:.4f}')
    ax.axvline(x=0, color='black', linewidth=0.8); ax.invert_yaxis()
    plt.tight_layout()
    out_dir = OUTPUTS / inst; out_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_dir / 'corr_problem_cluster.png', dpi=120)
    plt.show()

---
## Section 4 — Cluster-Level Importance (Headline Result)

### Clustered MDA
All columns in a cluster are permuted **simultaneously** with the same row shuffle. This destroys the cluster's shared information in one shot — no sibling can substitute. AUC drop = cluster's joint contribution.

Reported as mean ± std across 15 CPCV paths.

### Cross-checks
| Metric | What | Bias |
|---|---|---|
| **Clustered MDA** | Joint permutation AUC drop (out-of-sample) | Headline |
| **Clustered MDI** | Sum of per-feature tree impurity decrease | Train-set only ⚠ |
| **Group \|SHAP\|** | Sum of per-feature mean absolute SHAP values | Out-of-sample, tree-specific |

**Low-frequency fundamentals**: Any F11 macro cluster scoring highly is primarily a **print-day signal** — the model exploits discrete information arrival on EIA/PMI release days, not the continuous level.

In [ ]:
# Build clustered MDA + MDI + SHAP tables for all instruments
for inst in INSTRUMENTS:
    res = all_results[inst]
    imp = res['importance']
    cluster_map = res['cluster_map']

    cmdi  = {c: float(imp['mdi_mean'].reindex(cols).sum()) for c, cols in cluster_map.items()}
    cshap = {c: float(imp['shap_magnitude_mean'].reindex(cols).sum()) for c, cols in cluster_map.items()}

    cmda_df = pd.DataFrame({
        'MDA_mean':  imp['clustered_mda_mean'],
        'MDA_std':   imp['clustered_mda_std'],
        'MDI':       pd.Series(cmdi),
        'SHAP_mag':  pd.Series(cshap),
    })
    cmda_df.index.name = 'cluster'
    cmda_df = cmda_df.sort_values('MDA_mean', ascending=False)
    res['cmda_table'] = cmda_df

    out_dir = OUTPUTS / inst; out_dir.mkdir(parents=True, exist_ok=True)
    cmda_df.to_csv(out_dir / 'clustered_mda_full.csv')

print('Cluster importance tables built and saved.')

In [ ]:
# Per-instrument: full three-metric table with ranks
for inst in INSTRUMENTS:
    cmda_df = all_results[inst]['cmda_table'].copy()
    cmda_df['MDA_rank']  = cmda_df['MDA_mean'].rank(ascending=False).astype(int)
    cmda_df['MDI_rank']  = cmda_df['MDI'].rank(ascending=False).astype(int)
    cmda_df['SHAP_rank'] = cmda_df['SHAP_mag'].rank(ascending=False).astype(int)

    view = cmda_df[['MDA_mean','MDA_std','MDA_rank','MDI','MDI_rank','SHAP_mag','SHAP_rank']]

    print(f'\n{"="*65}')
    print(f'{inst}  —  cluster scores: MDA | MDI | SHAP  (with ranks)')
    print(f'{"="*65}')
    display(
        view.style
        .background_gradient(subset=['MDA_mean'], cmap='RdYlGn')
        .background_gradient(subset=['MDI'],      cmap='Blues')
        .background_gradient(subset=['SHAP_mag'], cmap='Purples')
        .format({'MDA_mean':'{:.4f}','MDA_std':'{:.4f}',
                 'MDI':'{:.4f}','SHAP_mag':'{:.4f}'})
    )

In [ ]:
# Three-metric comparison bar charts — one row per instrument
for inst in INSTRUMENTS:
    cmda_df = all_results[inst]['cmda_table'].sort_values('MDA_mean', ascending=True)
    n = len(cmda_df)
    fig, axes = plt.subplots(1, 3, figsize=(18, max(4, n * 0.38)))

    specs = [
        ('MDA_mean', 'MDA_std',  'Clustered MDA (AUC drop ± std)',   'RdYlGn'),
        ('MDI',      None,        'Clustered MDI (sum, train-bias ⚠)', 'Blues'),
        ('SHAP_mag', None,        'Group |SHAP| (sum)',                'Purples'),
    ]
    for ax, (col, err_col, title, cmap) in zip(axes, specs):
        vals  = cmda_df[col].values
        err   = cmda_df[err_col].fillna(0).values if err_col else None
        cmap_ = plt.get_cmap(cmap)
        base_c = cmap_(np.linspace(0.35, 0.75, n))
        bar_c  = ['tomato' if v < 0 else base_c[i] for i, v in enumerate(vals)]
        ax.barh(range(n), vals, xerr=err, align='center', color=bar_c, capsize=3, alpha=0.85)
        ax.set_yticks(range(n)); ax.set_yticklabels(cmda_df.index, fontsize=7)
        ax.set_xlabel(title, fontsize=9)
        ax.axvline(x=0, color='black', linewidth=0.8)

    fig.suptitle(f'{inst} — cluster importance: MDA vs MDI vs SHAP', fontsize=11, y=1.01)
    plt.tight_layout()
    out_dir = OUTPUTS / inst; out_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_dir / 'cluster_three_metrics.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved → {out_dir}/cluster_three_metrics.png')

In [ ]:
# Per-cluster feature-level breakdown: PFI, MDI, SHAP for every feature inside each cluster
# Sorted by cluster importance (MDA rank). Demonstrates the correlation-suppression effect.
for inst in INSTRUMENTS:
    res         = all_results[inst]
    cluster_map = res['cluster_map']
    cmda_df     = res['cmda_table']          # already sorted MDA descending
    imp         = res['importance']

    print(f'\n{"="*70}')
    print(f'{inst}  —  per-feature breakdown within each cluster')
    print(f'(clusters ordered by clustered MDA; features sorted by individual PFI)')
    print(f'{"="*70}')

    for cname in cmda_df.index:
        cols      = cluster_map.get(cname, [])
        mda_val   = cmda_df.loc[cname, 'MDA_mean']
        mda_std   = cmda_df.loc[cname, 'MDA_std']
        mdi_sum   = cmda_df.loc[cname, 'MDI']
        shap_sum  = cmda_df.loc[cname, 'SHAP_mag']

        feat_df = pd.DataFrame({
            'PFI (indiv.)':  imp['pfi_mean'].reindex(cols),
            'PFI_std':       imp['pfi_std'].reindex(cols),
            'MDI':           imp['mdi_mean'].reindex(cols),
            '|SHAP|':        imp['shap_magnitude_mean'].reindex(cols),
            'SHAP_signed':   imp['shap_signed_mean'].reindex(cols),
        }).dropna(how='all').sort_values('PFI (indiv.)', ascending=False)

        print(f'\n  {cname}  [{len(cols)} features]')
        print(f'  Clustered MDA = {mda_val:+.4f} ± {mda_std:.4f}  '
              f'| MDI sum = {mdi_sum:.4f}  | |SHAP| sum = {shap_sum:.4f}')
        display(
            feat_df.style
            .background_gradient(subset=['PFI (indiv.)'], cmap='Greens')
            .background_gradient(subset=['MDI'],          cmap='Blues')
            .background_gradient(subset=['|SHAP|'],       cmap='Purples')
            .format('{:.4f}')
        )

In [ ]:
# Cross-check: Spearman rank correlation between the three metric rankings
from scipy.stats import spearmanr

for inst in INSTRUMENTS:
    cmda_df = all_results[inst]['cmda_table']
    mda  = cmda_df['MDA_mean']
    mdi  = cmda_df['MDI']
    shap = cmda_df['SHAP_mag']

    r1, p1 = spearmanr(mda, mdi)
    r2, p2 = spearmanr(mda, shap)
    r3, p3 = spearmanr(mdi, shap)
    verdict = 'AGREE ✓' if (r1 > 0.5 and r2 > 0.5) else 'DISAGREE ✗'
    print(f'{inst}: MDA↔MDI ρ={r1:.2f}(p={p1:.3f})  '
          f'MDA↔SHAP ρ={r2:.2f}(p={p2:.3f})  '
          f'MDI↔SHAP ρ={r3:.2f}(p={p3:.3f})  → {verdict}')

In [ ]:
# Dimensionality-reduction check
print('Running dimensionality-reduction check (one medoid rep per cluster)...')
for inst in INSTRUMENTS:
    res = all_results[inst]
    dim_red = dimensionality_reduction_check(
        res['events_df'], res['feat_cols'], res['cluster_reps'], res['cluster_map'])
    res['dim_red'] = dim_red
    full_auc = np.mean(res['importance']['fold_aucs']) if res['importance']['fold_aucs'] else float('nan')
    print(f'{inst}: full ({len(res["feat_cols"])} feats) AUC={full_auc:.3f}  '
          f'reduced ({dim_red["n_reps"]} reps) AUC={dim_red["reduced_mean_auc"]:.3f} '
          f'(±{dim_red["reduced_std_auc"]:.3f})')

---
## Section 5 — Main Findings

- **Top cluster** = largest mean AUC drop (clustered MDA).
- **Pure cluster** = all features from one F-group; mixed = cross-group.
- **Print-day signal**: F11 macro cluster scoring highly → model exploits discrete release-day information arrival, not the continuous indicator level.

In [ ]:
# Main findings table
finding_rows = []
for inst in INSTRUMENTS:
    res     = all_results[inst]
    cmda_df = res['cmda_table']
    imp     = res['importance']
    dim_red = res.get('dim_red', {})

    full_auc   = np.mean(imp['fold_aucs']) if imp['fold_aucs'] else float('nan')
    top_c      = cmda_df.index[0]
    top_cols   = res['cluster_map'].get(top_c, [])
    pfx_counts = {}
    for c in top_cols:
        for p in CORR_CLUSTER_PREFIXES:
            if c.startswith(p):
                pfx_counts[p] = pfx_counts.get(p, 0) + 1; break
    dom = max(pfx_counts, key=pfx_counts.get) if pfx_counts else 'mixed'
    print_day = '⚠' if any('f11_' in c for c in top_cols) else ''

    finding_rows.append({
        'instrument':    inst,
        'full_AUC':      round(full_auc, 3),
        'reduced_AUC':   round(dim_red.get('reduced_mean_auc', float('nan')), 3),
        'n_clusters':    len(res['cluster_map']),
        'top_cluster':   top_c,
        'MDA_mean':      round(cmda_df['MDA_mean'].iloc[0], 4),
        'MDA_std':       round(cmda_df['MDA_std'].iloc[0], 4),
        'MDI_top':       round(cmda_df['MDI'].iloc[0], 4),
        'SHAP_top':      round(cmda_df['SHAP_mag'].iloc[0], 4),
        'dominant_Fg':   dom.rstrip('_'),
        'print_day':     print_day,
    })

findings_df = pd.DataFrame(finding_rows).set_index('instrument')
findings_df.to_csv(OUTPUTS / f'main_findings_{ASSET_CLASS}.csv')
print(f'Main findings — {ASSET_CLASS}:')
display(findings_df.T)

In [ ]:
# Top-5 cluster summary chart
fig, axes = plt.subplots(1, len(INSTRUMENTS), figsize=(6 * len(INSTRUMENTS), 5))
if len(INSTRUMENTS) == 1: axes = [axes]
for ax, inst in zip(axes, INSTRUMENTS):
    cmda_df = all_results[inst]['cmda_table']
    top5    = cmda_df.head(5).sort_values('MDA_mean', ascending=True)
    ax.barh(range(len(top5)), top5['MDA_mean'],
            xerr=top5['MDA_std'].fillna(0),
            color=['steelblue' if v >= 0 else 'tomato' for v in top5['MDA_mean']],
            align='center', capsize=3, alpha=0.85)
    ax.set_yticks(range(len(top5))); ax.set_yticklabels(top5.index, fontsize=8)
    ax.set_xlabel('Mean AUC drop (clustered MDA)')
    full_auc = np.mean(all_results[inst]['importance']['fold_aucs'])
    red_auc  = all_results[inst].get('dim_red', {}).get('reduced_mean_auc', float('nan'))
    ax.set_title(f'{inst}\nFull AUC={full_auc:.3f}  Reduced AUC={red_auc:.3f}', fontsize=9)
    ax.axvline(x=0, color='black', linewidth=0.8)
fig.suptitle(f'Top-5 clusters — {ASSET_CLASS} (clustered MDA)', fontsize=12, y=1.01)
plt.tight_layout()
fig.savefig(OUTPUTS / f'main_findings_top5_{ASSET_CLASS}.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Cluster purity vs F-groups
print(f'Cluster composition vs semantic F-groups — {ASSET_CLASS}')
for inst in INSTRUMENTS:
    print(f'\n{inst}:')
    for cname, cols in all_results[inst]['cluster_map'].items():
        pfx = {}
        for c in cols:
            for p in list(CORR_CLUSTER_PREFIXES) + ['f4_', 'f5_', 'f8_']:
                if c.startswith(p):
                    pfx[p] = pfx.get(p, 0) + 1; break
        top = sorted(pfx.items(), key=lambda x: -x[1])
        flag = '(PURE)' if len(pfx) <= 1 else '(MIXED)'
        print(f'  {cname:38s} n={len(cols):3d}  {flag}  '
              f'[{", ".join(f"{p.rstrip(chr(95))}:{n}" for p,n in top[:3])}]')

In [ ]:
# Print-day macro signal check
print('=== Print-day macro signal check ===')
for inst in INSTRUMENTS:
    res = all_results[inst]
    cmda_df = res['cmda_table']
    cluster_map = res['cluster_map']
    for rank, cname in enumerate(cmda_df.index, 1):
        cols = cluster_map.get(cname, [])
        f11_n  = sum(c.startswith('f11_') for c in cols)
        eia_pmi = [c for c in cols if any(k in c for k in ['stock_surprise','pmi','bdi'])]
        if f11_n:
            flag = '⚠ PRINT-DAY' if eia_pmi else '(level signal)'
            print(f'  {inst} | rank {rank:2d} | {cname} | MDA={cmda_df.loc[cname,"MDA_mean"]:+.4f} '
                  f'| F11 cols={f11_n} | {flag}')